In [0]:
VOLUME_PATH = "/Volumes/samplework/bronze/ecommerce_volume"

# Silver Streaming
SILVER_CHECKPOINT_ROOT = f"{VOLUME_PATH}/checkpoints/silver"

In [0]:
from pyspark.sql import functions as F

CATALOG = "samplework"

# ============================================================
# READ BRONZE TABLES
# ============================================================

customers_bronze = (
    spark.readStream
        .table(f"{CATALOG}.bronze.customers")
)

products_bronze = (
    spark.readStream
        .table(f"{CATALOG}.bronze.products")
)

orders_bronze = (
    spark.readStream
        .table(f"{CATALOG}.bronze.orders")
)

In [0]:
customers_clean = (
    customers_bronze
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("age", F.col("age").cast("integer"))
    .fillna({
        "city": "Unknown",
        "email": "unknown@email.com"
    })
    .filter(
        (F.col("age").isNull()) |
        ((F.col("age") >= 0) & (F.col("age") <= 120))
    )
    .dropDuplicates(["customer_id"])
)

In [0]:
# ============================================================
# PRODUCTS CLEANING
# ============================================================

products_clean = (
    products_bronze
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.upper(F.trim(F.col("category"))))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("stock", F.col("stock").cast("integer"))
    .fillna({
        "price": 0.0,
        "stock": 0,
        "category": "UNKNOWN"
    })
    .dropDuplicates(["product_id"])
)

In [0]:
valid_products = products_clean.filter(
    (F.col("price") >= 0) &
    (F.col("stock") >= 0)
)

In [0]:
# ============================================================
# ORDERS CLEANING
# ============================================================

orders_clean = (
    orders_bronze
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("quantity", F.col("quantity").cast("integer"))
    .withColumn("order_date", F.to_date(F.col("order_date")))
    .withColumn("status", F.upper(F.trim(F.col("status"))))
    .fillna({
        "quantity": 0,
        "status": "UNKNOWN"
    })
    .dropDuplicates(["order_id"])
)

In [0]:
# ============================================================
# VALID ORDERS
# ============================================================

valid_orders = orders_clean.filter(
    (F.col("quantity") > 0) &
    F.col("customer_id").isNotNull() &
    F.col("product_id").isNotNull() &
    F.col("order_date").isNotNull()
)

In [0]:
# ============================================================
# REJECTED / QUARANTINE ORDERS
# ============================================================

rejected_orders = orders_clean.filter(
    ~(
        (F.col("quantity") > 0) &
        F.col("customer_id").isNotNull() &
        F.col("product_id").isNotNull() &
        F.col("order_date").isNotNull()
    )
)


In [0]:
rejected_orders_query = (
    rejected_orders.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            f"{SILVER_CHECKPOINT_ROOT}/rejected_orders"
        )
        .trigger(availableNow=True)
        .toTable(f"{CATALOG}.silver.rejected_orders")
)

rejected_orders_query.awaitTermination()

In [0]:
# ============================================================
# DERIVED COLUMNS
# ============================================================

silver_orders = (
    valid_orders
    .withColumn(
        "total_amount",
        F.col("quantity") * F.col("price")
    )
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
    .withColumn("order_day", F.dayofmonth("order_date"))
    .withColumn(
        "order_status_group",
        F.when(F.col("status") == "COMPLETED", "SUCCESS")
         .when(F.col("status") == "CANCELLED", "CANCELLED")
         .otherwise("OTHER")
    )
)

In [0]:
# ============================================================
# COALESCE
# ============================================================

silver_orders = silver_orders.coalesce(4)

In [0]:


customers_query = (
    customers_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        f"{SILVER_CHECKPOINT_ROOT}/customers"
    )
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.silver.customers")
)

In [0]:
products_query = (
    valid_products.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        f"{SILVER_CHECKPOINT_ROOT}/products"
    )
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.silver.products")
)

In [0]:
orders_query = (
    valid_orders.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        f"{SILVER_CHECKPOINT_ROOT}/orders"
    )
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.silver.orders")
)